# Triton Puzzles — Part 1: Medium (warm-up to atomics)

**Puzzles 1–12.** Get the `pid → offs → mask` ritual into your fingers, then block reductions, then atomics. After this part you can fluently write any elementwise/reduction kernel.

Each puzzle: concept blurb → skeleton with `# TODO` → `test_pN()` that diffs against a torch reference. Solutions are at the bottom — don't peek until you've tried.

Inspired by Sasha Rush's GPU Puzzles, but for **Triton**.


## 0. The GPU mental model (read this once)

Before Triton makes sense, you need a picture of the machine.

```
         GPU
 ┌──────────────────────────────────────────────┐
 │  SM 0   SM 1   SM 2  ...  SM N                │   <- ~80–140 SMs on modern GPUs
 │  ┌──┐  ┌──┐                                   │
 │  │  │  │  │   each SM runs many WARPS (32     │
 │  │  │  │  │   threads in lockstep, SIMT)       │
 │  └──┘  └──┘                                   │
 │   |     |                                     │
 │   shared mem / L1   (~100KB per SM, fast)     │
 └──┬──────┬─────────────────────────────────────┘
    │      │
    └──────┴──── L2 cache (tens of MB, shared)
               │
               └── HBM / global memory (slow, GB)
```

**Triton's bargain with you**: you don't write per-thread code (like CUDA), you write per-*program* code. A program ≈ a CUDA block ≈ a tile of work. Inside, you operate on *vectors* (`BLOCK_SIZE` elements). The compiler vectorizes across threads, picks register allocation, decides shared-memory staging, and does software pipelining.

Key Triton primitives you'll use everywhere:
- `pid = tl.program_id(axis=0)` — which tile am I?
- `offs = pid * BLOCK + tl.arange(0, BLOCK)` — the row of indices this tile owns.
- `mask = offs < N` — guard for the tail tile.
- `x = tl.load(ptr + offs, mask=mask, other=0.0)` — vectorized gather from HBM.
- `tl.store(ptr + offs, val, mask=mask)` — vectorized scatter.

🧠 **Fun fact**: `tl.arange(0, BLOCK)` must have a `BLOCK` that is a *power of two known at compile time*. That's because Triton lowers the vector to a fixed-shape MLIR tensor; the compiler needs the shape to pick layouts. The same is why `BLOCK_SIZE` is a `tl.constexpr`.

In [ ]:
import os, math, time
import torch
import triton
import triton.language as tl

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    os.environ['TRITON_INTERPRET'] = '1'   # let kernels run on CPU for learning
    print('No GPU detected — using Triton interpreter mode. Slow but debuggable.')
else:
    print('Device:', torch.cuda.get_device_name(0))
    print('SMs   :', torch.cuda.get_device_properties(0).multi_processor_count)

torch.manual_seed(0)

def check(out, ref, atol=1e-3, rtol=1e-3, name=''):
    ok = torch.allclose(out, ref, atol=atol, rtol=rtol)
    diff = (out - ref).abs().max().item()
    print(f"{'✅' if ok else '❌'} {name}  max|Δ|={diff:.3e}")
    return ok


### A quick visualization of how `program_id` tiles a vector

```
 vector length N = 13,  BLOCK = 4   →  grid = ceil(13/4) = 4 programs

  index : 0  1  2  3 | 4  5  6  7 | 8  9 10 11 |12  X  X  X
  pid   :     0      |     1      |     2      |     3 (tail, masked)
```

Every program independently computes its slice. There is **no implicit communication between programs** — if you want a global reduction across tiles, you either do a second kernel or use atomics.

## Puzzle 1 — Constant fill
Write `1.0` into every position of an output tensor of length `N`. Practice the `pid → offs → mask` ritual.

In [ ]:
@triton.jit
def k_fill_ones(out_ptr, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    # TODO: store 1.0 into out_ptr at offs, masked
    pass

def test_p1():
    N = 1027
    out = torch.zeros(N, device=DEVICE)
    grid = (triton.cdiv(N, 128),)
    k_fill_ones[grid](out, N, BLOCK=128)
    check(out, torch.ones_like(out), name='P1 fill_ones')
test_p1()


## Puzzle 2 — Vector add
The hello-world. `z = x + y`. Pay attention: `tl.load` of a *masked* element returns `other` (default `0.0`), which is fine here because we only store back masked positions.

In [ ]:
@triton.jit
def k_add(x_ptr, y_ptr, z_ptr, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    # TODO: load x, y; add; store z
    pass

def test_p2():
    N = 100_000
    x = torch.randn(N, device=DEVICE); y = torch.randn(N, device=DEVICE)
    z = torch.empty_like(x)
    k_add[(triton.cdiv(N,1024),)](x, y, z, N, BLOCK=1024)
    check(z, x+y, name='P2 vector_add')
test_p2()


## Puzzle 3 — Scalar multiply with a runtime scalar
`y = alpha * x`. Note `alpha` is passed as a Python `float` (becomes a *kernel argument*, not a `constexpr`).

In [ ]:
@triton.jit
def k_scale(x_ptr, y_ptr, alpha, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid*BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    # TODO
    pass

def test_p3():
    N=10_000; x=torch.randn(N,device=DEVICE); y=torch.empty_like(x)
    k_scale[(triton.cdiv(N,256),)](x,y,2.5,N,BLOCK=256)
    check(y, 2.5*x, name='P3 scale')
test_p3()


## Puzzle 4 — ReLU
`y = max(x, 0)`. `tl.where(cond, a, b)` is your friend. Bonus: try `tl.maximum(x, 0.0)` — both compile to the same `max.f32` PTX.

In [ ]:
@triton.jit
def k_relu(x_ptr, y_ptr, N, BLOCK: tl.constexpr):
    pid = tl.program_id(0); offs = pid*BLOCK+tl.arange(0,BLOCK); mask = offs<N
    # TODO
    pass

def test_p4():
    N=5000; x=torch.randn(N,device=DEVICE); y=torch.empty_like(x)
    k_relu[(triton.cdiv(N,512),)](x,y,N,BLOCK=512)
    check(y, torch.relu(x), name='P4 relu')
test_p4()


## Puzzle 5 — Fused `a*x + b*y + bias`
Three loads, one store. The whole point of writing this in Triton instead of three torch ops is **memory-bandwidth fusion** — you touch HBM once per element instead of 3×.

In [ ]:
@triton.jit
def k_axpby(x_ptr,y_ptr,z_ptr,a,b,bias,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    # TODO
    pass

def test_p5():
    N=20_000; x=torch.randn(N,device=DEVICE); y=torch.randn(N,device=DEVICE); z=torch.empty_like(x)
    k_axpby[(triton.cdiv(N,512),)](x,y,z,1.3,-0.7,0.25,N,BLOCK=512)
    check(z, 1.3*x - 0.7*y + 0.25, name='P5 axpby')
test_p5()


## Puzzle 6 — Strided gather
`out[i] = x[i * stride]`. Now `offs` isn't the load index — you derive a *separate* set of pointers. This is where the pointer-arithmetic mental model starts.

In [ ]:
@triton.jit
def k_strided_gather(x_ptr,out_ptr,stride,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    # TODO: compute src_offs = offs * stride, load, store
    pass

def test_p6():
    M=10_000; stride=3; x=torch.arange(M*stride,device=DEVICE,dtype=torch.float32)
    out=torch.empty(M,device=DEVICE)
    k_strided_gather[(triton.cdiv(M,256),)](x,out,stride,M,BLOCK=256)
    check(out, x[::stride][:M], name='P6 strided_gather')
test_p6()


## Puzzle 7 — Reverse a vector
`out[i] = x[N-1-i]`. Easy logic, but a real-world reminder: the *load* pattern is now reversed — coalescing matters. On NVIDIA, a warp loading 32 contiguous floats issues **one** 128B transaction. A reversed pattern still coalesces (it's still contiguous, just backwards) but strided gathers do not.

In [ ]:
@triton.jit
def k_reverse(x_ptr,out_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    # TODO
    pass

def test_p7():
    N=8123; x=torch.randn(N,device=DEVICE); out=torch.empty_like(x)
    k_reverse[(triton.cdiv(N,256),)](x,out,N,BLOCK=256)
    check(out, torch.flip(x,(0,)), name='P7 reverse')
test_p7()


## Puzzle 8 — Clamp
`y = clamp(x, lo, hi)`. Compose `tl.maximum` / `tl.minimum`. (Triton does not yet have a single `tl.clamp`.)

In [ ]:
@triton.jit
def k_clamp(x_ptr,y_ptr,lo,hi,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    # TODO
    pass

def test_p8():
    N=3000; x=torch.randn(N,device=DEVICE)*3; y=torch.empty_like(x)
    k_clamp[(triton.cdiv(N,256),)](x,y,-1.0,1.0,N,BLOCK=256)
    check(y, x.clamp(-1,1), name='P8 clamp')
test_p8()


## Puzzle 9 — In-block cumulative sum
For a *single block* (assume `N <= BLOCK`), compute `out[i] = sum(x[:i+1])`. Use `tl.cumsum(x, axis=0)`.

In [ ]:
@triton.jit
def k_cumsum_one_block(x_ptr,out_ptr,N,BLOCK: tl.constexpr):
    offs = tl.arange(0,BLOCK); mask = offs<N
    # TODO: load x with other=0, cumsum, store
    pass

def test_p9():
    N=500; BLOCK=512
    x=torch.randn(N,device=DEVICE); out=torch.empty_like(x)
    k_cumsum_one_block[(1,)](x,out,N,BLOCK=BLOCK)
    check(out, torch.cumsum(x,0), atol=1e-2, name='P9 cumsum-1block')
test_p9()


## Puzzle 10 — Global sum reduction (two-pass)
Sum *all* of `x` (length `N`, arbitrary). Strategy: each program computes a partial, writes it to `partials[pid]`, then we reduce on the host (or with a second kernel).

In [ ]:
@triton.jit
def k_block_sum(x_ptr,partials_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    # TODO: load with other=0, sum to scalar, store at partials_ptr+pid
    pass

def test_p10():
    N=200_000; BLOCK=1024
    x=torch.randn(N,device=DEVICE); G=triton.cdiv(N,BLOCK)
    partials=torch.empty(G,device=DEVICE)
    k_block_sum[(G,)](x,partials,N,BLOCK=BLOCK)
    check(partials.sum().reshape(1), x.sum().reshape(1), atol=1e-2, name='P10 global_sum')
test_p10()


## Puzzle 11 — Global sum (one-pass with `tl.atomic_add`)
Same as P10 but instead of writing partials, every program atomically adds its block-sum to a single scalar. Faster for large `N`, but warning: floating-point atomics are **non-deterministic** in order, so you'll see tiny bit-level differences run to run.

In [ ]:
@triton.jit
def k_sum_atomic(x_ptr,out_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    # TODO: partial = sum(load(...)); tl.atomic_add(out_ptr, partial)
    pass

def test_p11():
    N=200_000; x=torch.randn(N,device=DEVICE)
    out=torch.zeros(1,device=DEVICE)
    k_sum_atomic[(triton.cdiv(N,1024),)](x,out,N,BLOCK=1024)
    check(out, x.sum().reshape(1), atol=1e-2, name='P11 atomic_sum')
test_p11()


## Puzzle 12 — Dot product (one-pass atomic)
Compute `sum(x * y)` for length-`N` vectors using the atomic trick.

In [ ]:
@triton.jit
def k_dot(x_ptr,y_ptr,out_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    # TODO
    pass

def test_p12():
    N=100_000; x=torch.randn(N,device=DEVICE); y=torch.randn(N,device=DEVICE)
    out=torch.zeros(1,device=DEVICE)
    k_dot[(triton.cdiv(N,1024),)](x,y,out,N,BLOCK=1024)
    check(out, (x*y).sum().reshape(1), atol=1e-1, rtol=1e-2, name='P12 dot')
test_p12()


# 🔒 SOLUTIONS — stop scrolling unless you tried!


In [ ]:
# P1
@triton.jit
def sol_p1(out_ptr, N, BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    tl.store(out_ptr+offs, tl.full((BLOCK,),1.0,tl.float32), mask=mask)

# P2
@triton.jit
def sol_p2(x_ptr,y_ptr,z_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    tl.store(z_ptr+offs, tl.load(x_ptr+offs,mask=mask) + tl.load(y_ptr+offs,mask=mask), mask=mask)

# P3
@triton.jit
def sol_p3(x_ptr,y_ptr,alpha,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    tl.store(y_ptr+offs, alpha*tl.load(x_ptr+offs,mask=mask), mask=mask)

# P4
@triton.jit
def sol_p4(x_ptr,y_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    tl.store(y_ptr+offs, tl.maximum(tl.load(x_ptr+offs,mask=mask), 0.0), mask=mask)

# P5
@triton.jit
def sol_p5(x_ptr,y_ptr,z_ptr,a,b,bias,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    x=tl.load(x_ptr+offs,mask=mask); y=tl.load(y_ptr+offs,mask=mask)
    tl.store(z_ptr+offs, a*x + b*y + bias, mask=mask)

# P6
@triton.jit
def sol_p6(x_ptr,out_ptr,stride,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    tl.store(out_ptr+offs, tl.load(x_ptr+offs*stride, mask=mask), mask=mask)

# P7
@triton.jit
def sol_p7(x_ptr,out_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    src = N-1-offs
    tl.store(out_ptr+offs, tl.load(x_ptr+src,mask=mask), mask=mask)

# P8
@triton.jit
def sol_p8(x_ptr,y_ptr,lo,hi,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    x = tl.load(x_ptr+offs,mask=mask)
    tl.store(y_ptr+offs, tl.minimum(tl.maximum(x,lo),hi), mask=mask)
# P9
@triton.jit
def sol_p9(x_ptr,out_ptr,N,BLOCK: tl.constexpr):
    offs=tl.arange(0,BLOCK); mask=offs<N
    x = tl.load(x_ptr+offs, mask=mask, other=0.0)
    tl.store(out_ptr+offs, tl.cumsum(x,axis=0), mask=mask)

# P10
@triton.jit
def sol_p10(x_ptr,partials_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    x = tl.load(x_ptr+offs, mask=mask, other=0.0)
    tl.store(partials_ptr+pid, tl.sum(x,0))

# P11
@triton.jit
def sol_p11(x_ptr,out_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    x = tl.load(x_ptr+offs, mask=mask, other=0.0)
    tl.atomic_add(out_ptr, tl.sum(x,0))

# P12
@triton.jit
def sol_p12(x_ptr,y_ptr,out_ptr,N,BLOCK: tl.constexpr):
    pid=tl.program_id(0); offs=pid*BLOCK+tl.arange(0,BLOCK); mask=offs<N
    x = tl.load(x_ptr+offs,mask=mask,other=0.0)
    y = tl.load(y_ptr+offs,mask=mask,other=0.0)
    tl.atomic_add(out_ptr, tl.sum(x*y, 0))


print('Solutions for puzzles 1–12 defined.')


## Next up

Open **Part 2** (`triton_puzzles_2_medium_hard.ipynb`) for 2D tiles, softmax, normalization, and your first matmul.
